# Model_KD_Scratch
Student trained from random initialisation.

Run **Model_VGG** and **Model_MobileNetV2** first.

In [ ]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/stm32-thesis/utils/")

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────
import os, time, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from utils.dataset import prepare_dataset, get_loaders
from utils.models  import VWW_MobileNetV2, VWW_VGGStyle
from utils.train   import setup_device, set_seed, evaluate, train_kd, plot_history

device = setup_device(seed=41)

In [ ]:
prepare_dataset()
train_loader, val_loader = get_loaders(batch_size=64, augmentation="standard")

In [ ]:
# ── Config ──────────────────────────────────────────────────────────
SAVE_DIR     = "/content/drive/My Drive/stm32-thesis/checkpoints"
TEACHER_CKPT = f"{SAVE_DIR}/vggstyle_seed_52.pth"     # adjust seed to your best
STUDENT_CKPT = f"{SAVE_DIR}/mobilenetv2_seed_85.pth"  # adjust seed to your best
KD_SAVE_PATH = f"{SAVE_DIR}/mobilenetv2_kd_scratch.pth"

KD_TEMPERATURE = 4.0
KD_ALPHA       = 0.7
KD_EPOCHS      = 30
KD_LR          = 1e-3

In [ ]:
# ── Load teacher and student ─────────────────────────────────────────
teacher = VWW_VGGStyle().to(device)
teacher.load_state_dict(torch.load(TEACHER_CKPT, map_location=device))

student = VWW_MobileNetV2().to(device)
print("✅ Student initialised from scratch")

# Sanity check — teacher must outperform student for KD to be useful
teacher_acc = evaluate(teacher, val_loader, device)
student_acc = evaluate(student, val_loader, device)
print(f"Teacher val: {teacher_acc*100:.2f}%")
print(f"Student val: {student_acc*100:.2f}%")
if teacher_acc <= student_acc:
    print("⚠️  WARNING: teacher is not stronger than student — KD may not help")

In [ ]:
# ── Run KD ──────────────────────────────────────────────────────────
best_val_acc, elapsed = train_kd(
    student      = student,
    teacher      = teacher,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    epochs       = KD_EPOCHS,
    lr           = KD_LR,
    weight_decay = 1e-4,
    temperature  = KD_TEMPERATURE,
    alpha        = KD_ALPHA,
    patience     = 10,
    save_path    = KD_SAVE_PATH,
)

In [ ]:
# ── Final val accuracy ───────────────────────────────────────────────
best_student = VWW_MobileNetV2().to(device)
best_student.load_state_dict(torch.load(KD_SAVE_PATH, map_location=device))
kd_val_acc = evaluate(best_student, val_loader, device)

print("="*50)
print(f"Teacher val acc  : {teacher_acc*100:.2f}%")
print(f"Student baseline : {student_acc*100:.2f}%")
print(f"KD best val acc  : {kd_val_acc*100:.2f}%")
print(f"Training time    : {elapsed:.1f} min")
print(f"Checkpoint       : {KD_SAVE_PATH}")
print("="*50)